### ingest_daily_support_tickets

In [1]:
# pip install boto3
# !pip install pymysql

import os
import pandas as pd
import pymysql
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ---------- CONFIG ----------
db_config = {
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "database": os.getenv("DB_NAME")
}
S3_BUCKET = "care-ticket-data" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [34]:
# ---------- UTILITY FUNCTIONS ----------

def get_engine(config):
    """Create a SQLAlchemy connection to the MySQL database."""
    return create_engine(
        f"mysql+pymysql://{config['user']}:{config['password']}@"
        f"{config['host']}:{config['port']}/{config['database']}"
    )


def upload_to_s3(df, bucket, key):
    """Convert a DataFrame to CSV and upload it to Amazon S3."""

    # Store CSV data in memory instead of creating a local file.
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    # Upload the CSV to the S3 raw layer.
    s3 = boto3.client("s3", **AWS_CONFIG)
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=csv_buffer.getvalue()
    )

    print(f"✅ Uploaded to s3://{bucket}/{key}")


def read_last_date(file_path):
    """Read the last processed date from the tracker file."""
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            return f.read().strip()

    # Starting point for the first pipeline run.
    return "2025-06-30"


def update_last_date(file_path, new_date):
    """Save the latest successfully processed date."""
    with open(file_path, "w") as f:
        f.write(new_date)


def get_next_date(last_date_str):
    """Calculate the next date to process."""
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)

    return next_date.strftime("%Y-%m-%d")


# ---------- MAIN INGESTION LOGIC ----------

def run_ingestion():
    """Extract the next day's support tickets and upload them to Amazon S3."""

    # Connect to the MySQL source database.
    engine = get_engine(db_config)

    # Determine the next date to process.
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Extract only records created on the next processing date.
    query = f"""
        SELECT *
        FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """

    df = pd.read_sql(query, engine)

    print(f"Records extracted: {len(df)}")

    # Skip the upload when no records exist for the date.
    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Build the destination path in the S3 raw layer.
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"

    upload_to_s3(
        df,
        S3_BUCKET,
        s3_key
    )

    # Update the tracker only after a successful upload.
    update_last_date(DATE_TRACKER_FILE, next_date)

    print(f"📅 Updated tracker to {next_date}")


if __name__ == "__main__":
    # Start one incremental ingestion cycle.
    run_ingestion()

Records extracted: 0
⚠️ No data found for 2025-08-01. Skipping upload.
